In [67]:
import keras
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

(60000, 28, 28) (60000,)
(10000, 28, 28) (10000,)


In [68]:
# turning big numbers into 0. numbers, for model to understand better
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

In [69]:
# splitting the training data into training and validation sets
rng = np.random.default_rng(42)
idx = rng.permutation(len(x_train))

x_val, y_val = x_train[idx[:10000]], y_train[idx[:10000]]
x_train, y_train = x_train[idx[10000:]], y_train[idx[10000:]]

print(x_train.shape, y_train.shape)

(50000, 28, 28) (50000,)


In [70]:
# reshaping the data to be 2D array for the model
# x_train_flat = x_train.reshape(-1, 28*28)
# x_val_flat = x_val.reshape(-1, 28*28)
# x_test_flat = x_test.reshape(-1, 28*28)

# converting the labels to int64 for compatibility with TensorFlow
y_train = y_train.astype('int64')
y_val = y_val.astype('int64')
y_test = y_test.astype('int64')

In [71]:
tf.random.set_seed(42)
# initializing weights and biases
# W1 = tf.Variable(tf.random.normal((784, 128)) * (2 / 784) ** 0.5)
#128 zeros for each of the 128 neurons in the hidden layer
# b1 = tf.Variable(tf.zeros(128))
# W2 = tf.Variable(tf.random.normal((128, 10)) * (2 / 128) ** 0.5)
# b2 = tf.Variable(tf.zeros(10))
# params = [W1, b1, W2, b2]

# keras creates weights inside the layers, so we no longer write tf.Variable ourselves.
model = keras.Sequential([
    keras.Input(shape=(28,28)),
    keras.layers.Reshape((28, 28,1)),
    keras.layers.Conv2D(32,3, activation='relu'),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(64,3, activation='relu'),
    keras.layers.MaxPooling2D(),
    keras.layers.Flatten(),
    keras.layers.Dense(10),
])
print(model.summary())



Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape_3 (Reshape)             │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_6 (Flatten)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 10)             │        16,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,826 (136.04 KB)

 Trainable params: 34,826 (136.04 KB)

 Non-trainable params: 0 (0.00 B)

None


In [72]:
# PART 3: THE NETWORK'S BUILDING BLOCKS
# Three functions we will use again and again:
#   forward     -> runs images through the network and gets 10 scores per image
#   loss_fn     -> measures how wrong those scores are (one number, lower = better)
#   accuracy_fn -> measures what fraction of images the network gets right

model.compile(
    optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

In [57]:
# PART 4: UNDERSTANDING CHECK
# Goal: calculate the gradient of W2 two ways (TensorFlow automatically, and by hand
# with a formula) and check that both give the same answer.
# Gradient = for every weight: "if I nudge it up a tiny bit, does the loss go up or down, and how much?"


In [73]:
history = model.fit(x_train, y_train, batch_size=64, epochs=5, validation_data=(x_val, y_val))
print(history.history.keys())

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9345 - loss: 0.2223 - val_accuracy: 0.9781 - val_loss: 0.0725
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9807 - loss: 0.0634 - val_accuracy: 0.9813 - val_loss: 0.0567
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9857 - loss: 0.0471 - val_accuracy: 0.9822 - val_loss: 0.0529
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9886 - loss: 0.0375 - val_accuracy: 0.9844 - val_loss: 0.0484
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9913 - loss: 0.0299 - val_accuracy: 0.9851 - val_loss: 0.0454
dict_keys(['accuracy', 'loss', 'val_accuracy', 'val_loss'])


In [74]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc:.4f}, Test loss: {test_loss:.4f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 999us/step - accuracy: 0.9892 - loss: 0.0327
Test accuracy: 0.9892, Test loss: 0.0327
